# 03 · Desafio 1 — Animais solitários ou em grupo?
**Pergunta:** qual é o número médio de indivíduos por registro de capivara?
A maioria dos registros envolve indivíduos isolados ou grupos?

In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.fauna.silver_registros")
capivara = silver.filter(F.col("especie") == "Capivara")

total = capivara.count()
print(f"Registros de capivara: {total}")

In [0]:
display(capivara.select(
    F.round(F.avg("individuos"), 2).alias("media"),
    F.median("individuos").alias("mediana"),
    F.min("individuos").alias("minimo"),
    F.max("individuos").alias("maximo"),
))

In [0]:
distribuicao = (
    capivara.groupBy("individuos")
    .count()
    .withColumn("percentual", F.round(F.col("count") / total * 100, 1))
    .orderBy("individuos")
)
display(distribuicao)

Databricks visualization. Run in Databricks to view.

In [0]:
import matplotlib.pyplot as plt
from pathlib import Path

pdf = distribuicao.toPandas()

fig, ax = plt.subplots(figsize=(8, 4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.bar(pdf["individuos"], pdf["count"], width=0.8,
       color="#2a78d6", edgecolor="#fcfcfb", linewidth=2, zorder=2)

ax.set_title("Capivara: a maioria dos registros tem de 4 a 7 indivíduos",
             loc="left", color="#0b0b0b", fontsize=12)
ax.set_xlabel("Indivíduos por registro", color="#52514e")
ax.set_ylabel("Quantidade de registros", color="#52514e")
ax.set_xticks(pdf["individuos"])
ax.tick_params(colors="#52514e")
ax.grid(axis="y", color="#e5e4e0", linewidth=0.8, zorder=0)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")

plt.tight_layout()

pasta_imagens = Path.cwd().parent / "imagens" 
pasta_imagens.mkdir(exist_ok=True) 
fig.savefig(pasta_imagens / "desafio1_distribuicao_capivara.png", dpi=150)

plt.show()

In [0]:
classificacao = (
    capivara
    .withColumn("tipo_registro", F.when(F.col("individuos") == 1, "Isolado (1)").otherwise("Grupo (2+)"))
    .groupBy("tipo_registro")
    .count()
    .withColumn("percentual", F.round(F.col("count") / total * 100, 1))
)
display(classificacao)

In [0]:
display(
    silver.groupBy("especie")
    .agg(
        F.round(F.avg("individuos"), 2).alias("media_individuos"),
        F.round(F.avg((F.col("individuos") == 1).cast("int")) * 100, 1).alias("pct_registros_isolados"),
    )
    .orderBy(F.desc("media_individuos"))
)

### Conclusão
- A capivara aparece em média com **5,95 indivíduos por registro** (mediana **6**). A média e a mediana
  são próximas, o que indica uma distribuição equilibrada.
- **99,2%** dos registros (725 de 731) envolvem grupos e apenas **0,8%** (6) mostram indivíduos isolados.
  Cerca de 65% dos registros têm entre 4 e 7 indivíduos.
- Entre as 13 espécies monitoradas, a capivara tem a maior média de indivíduos (a segunda, a ema, tem 3,57)
  e a menor proporção de registros isolados, em contraste com espécies como o tamanduá-bandeira
  (76,6% isolados) e o lobo-guará (74,8%).
- **Resposta:** a grande maioria dos registros de capivara envolve grupos, geralmente de 4 a 7 indivíduos.